<a href="https://colab.research.google.com/github/luaGeeko/Hands-On-Large-Language-Models/blob/main/Word2Vec_SkipGram%26CBOW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np
from typing import List, Tuple
from collections import Counter
import torch
from torch.utils.data import Dataset, DataLoader

random_seed = 42
np.random.seed(random_seed)
torch.manual_seed(random_seed)

In [2]:
corpus = [
  "the dog chased the cat",
  "the cat climbed the tree",
  "dogs and cats are good pets",
  "my dog likes to play with a ball",
  "the cat sleeps on the sofa",
  "a puppy is a young dog",
  "kittens are small and playful",
  "the pet store sells food for cats and dogs",
  "my dog loves running in the park",
  "cats enjoy sleeping in the sun",
]

In [3]:
# hyperparams
window_size = 2

# Build Vocab and Distribution - Tokenziation

In [4]:
# create vocabulary based on our corpus data
tokens = []
for sentence in corpus:
  tokens.extend(sentence.lower().split())
sorted_vocab = sorted(set(tokens))
print(f"built the vocab of size {len(sorted_vocab)}")

built the vocab of size 40


In [5]:
# compute distribution of the words
# we first to have how many times each word is present in our corpus data
word_frequency = Counter(tokens)
print(word_frequency)

Counter({'the': 9, 'dog': 4, 'cat': 3, 'and': 3, 'cats': 3, 'a': 3, 'dogs': 2, 'are': 2, 'my': 2, 'in': 2, 'chased': 1, 'climbed': 1, 'tree': 1, 'good': 1, 'pets': 1, 'likes': 1, 'to': 1, 'play': 1, 'with': 1, 'ball': 1, 'sleeps': 1, 'on': 1, 'sofa': 1, 'puppy': 1, 'is': 1, 'young': 1, 'kittens': 1, 'small': 1, 'playful': 1, 'pet': 1, 'store': 1, 'sells': 1, 'food': 1, 'for': 1, 'loves': 1, 'running': 1, 'park': 1, 'enjoy': 1, 'sleeping': 1, 'sun': 1})


In [6]:
# now we need the mapping dict from word --> id and vice versa
vocab_to_id = {word: idx for idx, word in enumerate(sorted_vocab)}
id_to_vocab = {idx: word for idx, word in enumerate(sorted_vocab)}

In [7]:
# Generate center_id and context_id pairs as per window size
center_context_pairs = []
for t, center_word in enumerate(tokens):
  context_words = []
  window_start = max(0, t - window_size)
  window_end = min(len(tokens), t+ window_size+1)
  for context_id in range(window_start, window_end):
    if context_id == t:
      continue
    # make center_id and context_id pair tuple
    center_context_pairs.append((center_word, tokens[context_id]))
print(f"created {len(center_context_pairs)} center word and context word pairs from the tokens | sample pairs {center_context_pairs[0:10]}")

created 246 center word and context word pairs from the tokens | sample pairs [('the', 'dog'), ('the', 'chased'), ('dog', 'the'), ('dog', 'chased'), ('dog', 'the'), ('chased', 'the'), ('chased', 'dog'), ('chased', 'the'), ('chased', 'cat'), ('the', 'dog')]


# Creating Dataset Loader for small dataset

In [8]:
class CorpusDataset(Dataset):
  """ our small sample corpus dataset """
  def __init__(self, dataset_pairs: List[Tuple[str, str]]):
    self.pairs = dataset_pairs

  def __len__(self):
    return len(self.pairs)

  def __getitem__(self, idx):
    return self.pairs[idx]

def collate_fn(data: List[Tuple[str]]):
  center_words, context_words = zip(*data)
  center_ids = torch.tensor([vocab_to_id[w] for w in center_words], dtype=torch.long)
  context_ids = torch.tensor([vocab_to_id[w] for w in context_words], dtype=torch.long)

In [9]:
dataloader = DataLoader(dataset=CorpusDataset(dataset_pairs=center_context_pairs), batch_size=4, collate_fn=collate_fn)